# 01 - Data pipeline

Audit and clean the configured BTC-USDT dataset, then save reusable Parquet data and a small set of exploratory figures. Run notebooks **01 - 04** in order. The 800,000-row cap is intentional; raise it only when memory and Spark capacity permit.

In [1]:
## Setup
import os
os.environ.pop("SPARK_HOME", None)  # Use the JARs bundled with the installed PySpark version.

import sys, yaml
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import plotly.graph_objects as go

_cwd = Path.cwd().resolve()
_candidates = list(dict.fromkeys([
    _cwd,
    *_cwd.parents,
    *(path.parent.parent for path in _cwd.glob("*/configs/pipeline.yaml")),
]))
_required_project_files = (
    "configs/pipeline.yaml",
    "src/features/pipeline.py",
    "notebooks/01_data_pipeline.ipynb",
    "Home.py",
)
_project_roots = [
    path for path in _candidates
    if all((path / relative).is_file() for relative in _required_project_files)
]
if not _project_roots:
    raise FileNotFoundError(
        "Could not identify this project. Open its repository root or set the notebook working directory to it."
    )
if len(_project_roots) > 1:
    raise RuntimeError(
        f"Multiple project roots found: {_project_roots}. Open the intended repository root."
    )
PROJECT_ROOT = _project_roots[0]
os.chdir(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.spark_session import get_spark
from src.data.loader        import primary_to_pandas
from src.data.profiler      import profile_dataframe
from src.data.cleaner       import standardise_timestamps, clean_and_align, report_gaps
from src.visualisation.eda_plots import plot_price_volume, plot_returns_distribution

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)
java_home = CFG.get("spark", {}).get("java_home")
if java_home and not Path(java_home).is_dir():
    raise FileNotFoundError(f"Configured spark.java_home does not exist: {java_home}. Update configs/pipeline.yaml.")

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
for d in [PROCESSED, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print('Setup complete.')

Project root : C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis
Setup complete.


In [2]:
# ── 1b. Materialise full primary dataset ─────────────────────────────
print('Loading primary dataset (up to 800,000 rows)...')
df_primary = primary_to_pandas(max_rows=800_000)
print(f'Sample shape: {df_primary.shape}')
sample = df_primary.head(10_000)
null_pct = (sample.isnull().sum() / len(sample) * 100).round(2)
print(f'\nNull % (sample):')
flagged = null_pct[null_pct > 0]
print(flagged.to_string() if len(flagged) else '  None.')
print(f'Duplicates (sample): {sample.duplicated().sum():,}')

Loading primary dataset (up to 800,000 rows)...


Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

Sample shape: (800000, 23)

Null % (sample):
  None.
Duplicates (sample): 0


In [3]:
# ── 1c. Timestamp detection & coverage ───────────────────────────────
TS_CANDIDATES = [c for c in df_primary.columns
                 if c.lower() in ('timestamp','date','open_time','time','close_time')]
TS_COL = TS_CANDIDATES[0] if TS_CANDIDATES else df_primary.columns[0]
print(f'Timestamp column : "{TS_COL}"')

try:
    df_primary[TS_COL] = pd.to_datetime(df_primary[TS_COL], unit='ms', utc=True)
except Exception:
    df_primary[TS_COL] = pd.to_datetime(df_primary[TS_COL], utc=True)

df_primary = df_primary.sort_values(TS_COL)
median_delta    = df_primary[TS_COL].diff().median()
EXPECTED_SECS   = int(median_delta.total_seconds()) if hasattr(median_delta,'total_seconds') else 3600

print(f'Date range  : {df_primary[TS_COL].min()}  →  {df_primary[TS_COL].max()}')
print(f'Total bars  : {len(df_primary):,}')
print(f'Bar interval: {median_delta}  ({EXPECTED_SECS}s)')

Timestamp column : "timestamp"
Date range  : 2017-08-18 01:19:00+00:00  →  2020-05-14 07:51:00+00:00
Total bars  : 800,000
Bar interval: 0 days 00:01:00  (60s)


In [4]:
# ── 1d. Tick / order-book column check ───────────────────────────────
TICK_INDICATORS = [
    'taker_buy_base_asset_volume','taker_buy_quote_asset_volume',
    'number_of_trades','trade_id','aggressor_side','bid_volume','ask_volume',
]
found_tick_cols = [c for c in df_primary.columns if c in TICK_INDICATORS]
HAS_TICK_DATA   = len(found_tick_cols) > 0

print(f'Tick / order-book columns : {found_tick_cols}')
print(f'Hawkes / OFI / VPIN active: {HAS_TICK_DATA}')
if not HAS_TICK_DATA:
    print('  → Conditional microstructure features are OUT OF SCOPE for this dataset.')

Tick / order-book columns : []
Hawkes / OFI / VPIN active: False
  → Conditional microstructure features are OUT OF SCOPE for this dataset.


In [5]:
# ── 1f. Save audit metadata ───────────────────────────────────────────
OHLC_COLS  = [c for c in ['open','high','low','close'] if c in df_primary.columns]
audit_meta = {
    'ts_col'                    : TS_COL,
    'has_tick_data'             : bool(HAS_TICK_DATA),
    'tick_columns'              : found_tick_cols,
    'expected_interval_seconds' : EXPECTED_SECS,
    'date_min'                  : str(df_primary[TS_COL].min()),
    'date_max'                  : str(df_primary[TS_COL].max()),
    'total_bars'                : int(len(df_primary)),
    'ohlc_columns'              : OHLC_COLS,
    'volume_col'                : 'volume' if 'volume' in df_primary.columns else None,
    'null_pct'                  : null_pct.to_dict(),
}
meta_path = PROCESSED / 'audit_meta.yaml'
with open(meta_path, 'w') as f:
    yaml.dump(audit_meta, f, default_flow_style=False)
print(f'Audit metadata saved → {meta_path}')

Audit metadata saved → C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis\processed\audit_meta.yaml


---
## 2 · PySpark Ingestion & Cleaning

In [6]:
# ── 2a. Start Spark ───────────────────────────────────────────────────
spark = get_spark()
print(f'Spark {spark.version} started.')

[spark_session] JAVA_HOME pinned → C:\Program Files\Java\jdk-17
Spark 4.2.0 started.


In [7]:
# ── 2b. Load full dataset → Spark DataFrame ──────────────────────────

print('Reusing the audited primary dataset (no second download)…')
df_full = df_primary
print(f'Full dataset shape: {df_full.shape}')

# The timestamp column is a datetime string ('YYYY-MM-DD HH:MM:SS').
# standardise_timestamps() auto-detects the string type and uses
# F.to_timestamp() — no epoch-ms conversion needed here.
sdf = spark.createDataFrame(df_full)
print(f'Spark DataFrame : {sdf.count():,} rows × {len(sdf.columns)} columns')
sdf.printSchema()

Reusing the audited primary dataset (no second download)…
Full dataset shape: (800000, 23)
Spark DataFrame : 800,000 rows × 23 columns
root
 |-- timestamp: timestamp (nullable = true)
 |-- open: string (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: string (nullable = true)
 |-- MA_20: double (nullable = true)
 |-- MA_50: double (nullable = true)
 |-- MA_200: double (nullable = true)
 |-- RSI: double (nullable = true)
 |-- %K: double (nullable = true)
 |-- %D: double (nullable = true)
 |-- ADX: double (nullable = true)
 |-- ATR: double (nullable = true)
 |-- Trendline: double (nullable = true)
 |-- MACD: double (nullable = true)
 |-- BL_Upper: double (nullable = true)
 |-- BL_Lower: double (nullable = true)
 |-- Signal: double (nullable = true)
 |-- Histogram: double (nullable = true)
 |-- MN_Upper: double (nullable = true)
 |-- MN_Lower: double (nullable = true)
 |-- month: long (nullable = true

In [8]:
# ── 2c. Profile raw data ──────────────────────────────────────────────
_ = profile_dataframe(sdf, name='Primary Raw')


  PROFILE: Primary Raw
  Rows        : 800,000
  Columns     : 23

  Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- open: string (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: string (nullable = true)
 |-- MA_20: double (nullable = true)
 |-- MA_50: double (nullable = true)
 |-- MA_200: double (nullable = true)
 |-- RSI: double (nullable = true)
 |-- %K: double (nullable = true)
 |-- %D: double (nullable = true)
 |-- ADX: double (nullable = true)
 |-- ATR: double (nullable = true)
 |-- Trendline: double (nullable = true)
 |-- MACD: double (nullable = true)
 |-- BL_Upper: double (nullable = true)
 |-- BL_Lower: double (nullable = true)
 |-- Signal: double (nullable = true)
 |-- Histogram: double (nullable = true)
 |-- MN_Upper: double (nullable = true)
 |-- MN_Lower: double (nullable = true)
 |-- month: long (nullable = true)

  Null percentages:
    timestamp                     

In [9]:
# ── 2d. Standardise + Clean ───────────────────────────────────────────
sdf_clean = standardise_timestamps(sdf, ts_col=TS_COL)
for column in ('open', 'high', 'low', 'close', 'volume'):
    if column in sdf_clean.columns:
        sdf_clean = sdf_clean.withColumn(column, F.col(column).cast('double'))
sdf_clean = sdf_clean.dropna(subset=[TS_COL, 'open', 'high', 'low', 'close'])
sdf_clean = sdf_clean.filter(F.col('close') > 0)
sdf_clean = clean_and_align(sdf_clean, ts_col=TS_COL)

n_before = sdf.count()
n_after  = sdf_clean.count()
print(f'After cleaning: {n_after:,} rows  (dropped {n_before - n_after:,})')


After cleaning: 800,000 rows  (dropped 0)


In [10]:
# ── 2e. Timestamp gap report ─────────────────────────────────────────
gaps_sdf  = report_gaps(sdf_clean, ts_col=TS_COL, expected_interval_seconds=EXPECTED_SECS)
gap_count = gaps_sdf.count()
print(f'Gaps larger than {EXPECTED_SECS}s : {gap_count:,}')
if gap_count > 0:
    gaps_sdf.orderBy('gap_seconds', ascending=False).show(20, truncate=False)

Gaps larger than 60s : 1,014
+-------------------+-------------------+-----------+
|gap_start          |gap_end            |gap_seconds|
+-------------------+-------------------+-----------+
|2018-02-07 21:09:00|2018-02-09 18:49:00|164400     |
|2018-07-03 20:09:00|2018-07-04 14:26:00|65820      |
|2018-11-13 21:09:00|2018-11-14 14:49:00|63600      |
|2019-03-11 21:09:00|2019-03-12 13:49:00|60000      |
|2019-11-12 21:09:00|2019-11-13 10:09:00|46800      |
|2019-11-24 21:09:00|2019-11-25 09:49:00|45600      |
|2017-09-06 20:09:00|2017-09-07 07:49:00|42000      |
|2017-10-29 20:09:00|2017-10-30 07:49:00|42000      |
|2018-10-28 20:09:00|2018-10-29 07:49:00|42000      |
|2019-10-27 20:09:00|2019-10-28 07:49:00|42000      |
|2020-02-08 21:09:00|2020-02-09 08:49:00|42000      |
|2019-06-07 20:09:00|2019-06-08 07:04:00|39300      |
|2017-09-28 20:09:00|2017-09-29 06:49:00|38400      |
|2017-09-09 20:09:00|2017-09-10 06:49:00|38400      |
|2017-09-08 20:09:00|2017-09-09 06:49:00|38400      |

In [11]:
# ── 2f. Collect to pandas + save ─────────────────────────────────────
df_clean = sdf_clean.toPandas()
df_clean[TS_COL] = pd.to_datetime(df_clean[TS_COL])
df_clean = df_clean.sort_values(TS_COL).reset_index(drop=True)

out_path = PROCESSED / 'btc_cleaned.parquet'
df_clean.to_parquet(out_path, index=False)
print(f'Saved → {out_path}  ({df_clean.shape})')

spark.stop()
print('Spark stopped.')

Saved → C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis\processed\btc_cleaned.parquet  ((800000, 23))
Spark stopped.


---
## 3 · Exploratory Data Analysis (EDA)

In [12]:
# ── 3a. Price & Volume ────────────────────────────────────────────────
fig = plot_price_volume(df_clean, ts_col=TS_COL)
fig.write_html(str(FIGURES / 'price_volume.html'), include_plotlyjs='cdn')


In [13]:
# ── 3b. Log-return descriptives ───────────────────────────────────────
df_clean['log_return'] = np.log(df_clean['close'] / df_clean['close'].shift(1)).replace([np.inf, -np.inf], np.nan)
returns = df_clean['log_return'].dropna()

print(returns.describe().round(6))
print(f'Skewness : {returns.skew():.4f}  |  Kurtosis (excess): {returns.kurtosis():.4f}')

fig = plot_returns_distribution(df_clean)
fig.write_html(str(FIGURES / 'return_distribution.html'), include_plotlyjs='cdn')


count    799999.000000
mean          0.000001
std           0.001903
min          -0.406772
25%          -0.000379
50%           0.000000
75%           0.000379
max           0.212098
Name: log_return, dtype: float64
Skewness : -10.3348  |  Kurtosis (excess): 3331.7246


In [14]:
# ── 3c. Volatility clustering + rolling vol ───────────────────────────
fig = go.Figure(go.Scatter(
    x=df_clean[TS_COL], y=df_clean['log_return']**2,
    line=dict(color='#F7931A', width=0.7), name='Squared Return',
))
fig.update_layout(title='Volatility Clustering (Squared Log Returns)',
                  template='plotly_dark', yaxis_title='Squared Return')
fig.write_html(str(FIGURES / 'vol_clustering.html'), include_plotlyjs='cdn')

fig2 = go.Figure()
for w, col in [(7,'#4FC3F7'), (30,'#F7931A'), (90,'#66BB6A')]:
    fig2.add_trace(go.Scatter(x=df_clean[TS_COL], y=df_clean['log_return'].rolling(w).std(),
                              name=f'{w}-bar vol', line=dict(color=col)))
fig2.update_layout(title='Rolling Volatility', template='plotly_dark')
fig2.write_html(str(FIGURES / 'rolling_vol_eda.html'), include_plotlyjs='cdn')


In [ ]:
# ── 3d. ACF / PACF ───────────────────────────────────────────────────
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
plot_acf(returns,     lags=50, ax=axes[0,0], title='ACF – Log Returns')
plot_pacf(returns,    lags=50, ax=axes[0,1], title='PACF – Log Returns')
plot_acf(returns**2,  lags=50, ax=axes[1,0], title='ACF – Squared Returns')
plot_pacf(returns**2, lags=50, ax=axes[1,1], title='PACF – Squared Returns')
plt.tight_layout()
plt.savefig(str(FIGURES / 'acf_pacf.png'), dpi=120, bbox_inches='tight')
plt.close()

print('EDA complete.')